# VINCI Embodied AI — From gesture to mission

This notebook is in English. Run cells in order with **Shift + Enter**.

**Start again from zero:** use the **RESTART WORKSHOP** desktop icon. This archives your work and resets the workshop; it is not needed to refresh the language.

In [ ]:
LANG = "en"   # Notebook language: English
from IPython.display import Markdown, display
T = lambda fr, en: display(Markdown(en if LANG == "en" else fr))   # notebook texts
t = lambda fr, en: en if LANG == "en" else fr                      # code messages
print("language: English" if LANG != "en" else "language: English")

In [ ]:
T(r"""
RL, imitation, perception, then orchestration of a robotic mission.

| Workshop · 2 h | |
|---|---|
| 0–10 min | Facilitator introduction |
| 10–15 min | Group quiz: three questions |
| 15–60 min | Login and notebook |
| 60–115 min | Challenge in `team/`, including submission |
| 115–120 min | Results and discussion |

Section timings below are relative to the start of the notebook.

Run cells in order with **Shift + Enter**. `[*]` means a cell is working: wait for completion before running it again.

**Where should you look?**
- **Training**: follow the notebook output and curve. Training simulations run separately; the robot in the Isaac window may stay still.
- **Reference**: watch the notebook tables and videos, recorded before the workshop.
- **Live robot**: watch Isaac Sim for the drawer; RAM images and video appear in the notebook.

A settings cell prepares parameters; the next cell starts computation. Each section identifies what happens next.
""",
r"""
RL, imitation, perception, then orchestration of a robotic mission.

| Workshop · 2 h | |
|---|---|
| 0–10 min | Facilitator introduction |
| 10–15 min | Group quiz: three questions |
| 15–60 min | Login and notebook |
| 60–115 min | Challenge in `team/`, including submission |
| 115–120 min | Results and discussion |

Section timings below are relative to the start of the notebook.

Run cells in order with **Shift + Enter**. `[*]` means a cell is working: wait for completion before running it again.

**Where should you look?**
- **Training**: follow the notebook output and curve. Training simulations run separately; the robot in the Isaac window may stay still.
- **Reference**: watch the notebook tables and videos, recorded before the workshop.
- **Live robot**: watch Isaac Sim for the drawer; RAM images and video appear in the notebook.

A settings cell prepares parameters; the next cell starts computation. Each section identifies what happens next.
""")

RL, imitation, perception, then orchestration of a robotic mission.

| Workshop · 2 h | |
|---|---|
| 0–10 min | Facilitator introduction |
| 10–15 min | Group quiz: three questions |
| 15–60 min | Login and notebook |
| 60–115 min | Challenge in `team/`, including submission |
| 115–120 min | Results and discussion |

Section timings below are relative to the start of the notebook.

Run cells in order with **Shift + Enter**. `[*]` means a cell is working: wait for completion before running it again.

**Where should you look?**
- **Training**: follow the notebook output and curve. Training simulations run separately; the robot in the Isaac window may stay still.
- **Reference**: watch the notebook tables and videos, recorded before the workshop.
- **Live robot**: watch Isaac Sim for the drawer; RAM images and video appear in the notebook.

A settings cell prepares parameters; the next cell starts computation. Each section identifies what happens next.

In [ ]:
import os, sys, json, time, glob, re, subprocess
import yaml
import matplotlib.pyplot as plt
from IPython.display import Video
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()   # your workspace
RT = os.environ.get("VINCI_RUNTIME", "/opt/vinci-workshop")                                       # simulator and models
os.environ.setdefault("VINCI_WORKSPACE", ROOT)
sys.path[:0] = [RT, os.path.join(RT, "lib")]
from vinci_lab.client import Lab
import timeline
A = os.path.join(RT, "assets")                      # reference results supplied with the workshop
QUICK = os.environ.get("VINCI_NB_FAST") == "1"      # short mode used by automated machine checks
def tl(event, **kw):                                  # journey log (in Jupyter Lab only, not automated tests)
    if os.environ.get("JPY_SESSION_NAME"):
        timeline.log(event, **kw)
tl("notebook_start", lang=LANG)

def run(cmd, keep=None, every=1, always=None):
    # Run a workshop command and stream the relevant output lines.
    env = {k: v for k, v in os.environ.items() if k != "MPLBACKEND"}   # Jupyter’s inline backend is incompatible with Isaac
    p = subprocess.Popen(["bash", "-c", f"cd {ROOT} && {cmd}"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         stdin=subprocess.DEVNULL, text=True, env=env)
    lines, n = [], 0
    for line in p.stdout:
        line = re.sub(r"\x1b\[[0-9;]*m", "", line).strip()
        lines.append(line + "\n")
        if always and re.search(always, line):
            print(line[:160]); continue
        if keep is None or re.search(keep, line):
            n += 1
            if n % every == 0 or keep is None:
                print(line[:160])
    p.wait()
    return "".join(lines)

lab = Lab()                                          # simulator (Isaac window): 1 robot, 1 cube, 1 drawer
print(t("simulator:", "simulator:"), "OK" if "success" in lab.success() else lab.success())

In [ ]:
T(r"""
## 1. Observe and act · 8 min

**Live observation — look at the notebook images and the Isaac Sim scene.**

A Franka, a cube, a drawer, two cameras: fixed and wrist-mounted. Here, `observe()` exposes the images **and the exact simulator state**, including cube and gripper positions. An observation is the information available for making a decision; an action is a robot command. Nothing is trained in this first manipulation.
""",
r"""
## 1. Observe and act · 8 min

**Live observation — look at the notebook images and the Isaac Sim scene.**

A Franka, a cube, a drawer, two cameras: fixed and wrist-mounted. Here, `observe()` exposes the images **and the exact simulator state**, including cube and gripper positions. An observation is the information available for making a decision; an action is a robot command. Nothing is trained in this first manipulation.
""")

## 1. Observe and act · 8 min

**Live observation — look at the notebook images and the Isaac Sim scene.**

A Franka, a cube, a drawer, two cameras: fixed and wrist-mounted. Here, `observe()` exposes the images **and the exact simulator state**, including cube and gripper positions. An observation is the information available for making a decision; an action is a robot command. Nothing is trained in this first manipulation.

In [ ]:
tl("first_action")
o = lab.observe(); lab.show(o)
print(t("gripper:", "gripper:"), o["ee_pos"], t("| cube:", "| cube:"), o["cube_pos"], t("| drawer open by", "| drawer open by"), o["drawer_opening_m"], "m")
print("images :", o["rgb_front"].shape, o["rgb_wrist"].shape, t("-> o['rgb_front'], o['rgb_wrist']: usable in your own code",
                                                                "-> o['rgb_front'], o['rgb_wrist']: usable in your own code"))

In [ ]:
T(r"""
### Geometric control

**Live robot — watch Isaac Sim during the next cell.**

Approach, descend, close the gripper, lift. This programmed sequence uses the exact cube position: **privileged state**, without visual estimation. It does not update network weights.
""",
r"""
### Geometric control

**Live robot — watch Isaac Sim during the next cell.**

Approach, descend, close the gripper, lift. This programmed sequence uses the exact cube position: **privileged state**, without visual estimation. It does not update network weights.
""")

### Geometric control

**Live robot — watch Isaac Sim during the next cell.**

Approach, descend, close the gripper, lift. This programmed sequence uses the exact cube position: **privileged state**, without visual estimation. It does not update network weights.

In [ ]:
import math
lab.reset(); c = lab.observe()["cube_pos"]; yaw = math.pi / 2; q = (0.0, math.cos(yaw / 2), math.sin(yaw / 2), 0.0)
for name, step in [(t("above the cube", "above the cube"), lambda: lab.move_to([c[0], c[1], c[2] + 0.15], q)),
                   (t("open the gripper", "open the gripper"),  lambda: lab.gripper(open=True)),
                   (t("go down", "go down"),                 lambda: lab.move_to([c[0], c[1], c[2] + 0.005], q)),
                   (t("close the gripper", "close the gripper"), lambda: lab.gripper(open=False)),
                   (t("lift", "lift"),                        lambda: lab.move_to([c[0], c[1], c[2] + 0.20], q))]:
    r = step(); print(f"{name:18s} ok={r['ok']}  {r['reason']}")
print(t("cube now:", "cube now:"), lab.observe()["cube_pos"])

In [ ]:
T(r"""
### Evaluate a mission

**Evaluation — read the notebook results; the Isaac window does not show these episodes.**

The starter agent puts away several parts, closes the drawer and must preserve a gauge. Evaluation over **3 episodes, about 1 min**. This measures existing behaviour without training models.

`Complete successes` measures final task success; `Average score` also includes intermediate steps and checks. A successful mission can therefore score below 100/100.
""",
r"""
### Evaluate a mission

**Evaluation — read the notebook results; the Isaac window does not show these episodes.**

The starter agent puts away several parts, closes the drawer and must preserve a gauge. Evaluation over **3 episodes, about 1 min**. This measures existing behaviour without training models.

`Complete successes` measures final task success; `Average score` also includes intermediate steps and checks. A successful mission can therefore score below 100/100.
""")

### Evaluate a mission

**Evaluation — read the notebook results; the Isaac window does not show these episodes.**

The starter agent puts away several parts, closes the drawer and must preserve a gauge. Evaluation over **3 episodes, about 1 min**. This measures existing behaviour without training models.

`Complete successes` measures final task success; `Average score` also includes intermediate steps and checks. A successful mission can therefore score below 100/100.

In [ ]:
out = run("./evaluate.sh --episodes 3 --procs 3", keep=r"^(Episode|Complete|Average|Failures|- )")

In [ ]:
T(r"""
## 2. Learn a gesture · 18 min

### 2.1 RL: a privileged expert

**Actual training — follow the notebook output and curve. Isaac Sim may stay still.**

In reinforcement learning (**RL**), a **policy** maps observations to actions. The simulator computes a **reward**: approaching the handle, aligning the gripper, grasping and opening earn reward here; abrupt movements are penalized. The network learns to increase the sum of future rewards.

An **environment** is a simulated copy of the problem; an **episode** is an attempt between resets. The 256 environments collect experience for one shared policy. It receives joint and drawer state and the relative handle position, **not images**.

**PPO** alternates experience collection and weight updates. The actor chooses actions; the critic estimates future rewards. Actions doing better than expected become more likely, with an objective that limits excessive changes. Exploration tries other actions, while precise control requires reducing unnecessary randomness.

The first cell sets `num_envs` (parallel experience collection) and `max_iterations` (collection and optimization cycles). The next starts **3–4 min** of computation, including evaluation and export. More environments also increase the data budget per iteration; more iterations do not guarantee a better result.

The curve measures **reward**, not success rate. A separate evaluation measures drawer opening.
""",
r"""
## 2. Learn a gesture · 18 min

### 2.1 RL: a privileged expert

**Actual training — follow the notebook output and curve. Isaac Sim may stay still.**

In reinforcement learning (**RL**), a **policy** maps observations to actions. The simulator computes a **reward**: approaching the handle, aligning the gripper, grasping and opening earn reward here; abrupt movements are penalized. The network learns to increase the sum of future rewards.

An **environment** is a simulated copy of the problem; an **episode** is an attempt between resets. The 256 environments collect experience for one shared policy. It receives joint and drawer state and the relative handle position, **not images**.

**PPO** alternates experience collection and weight updates. The actor chooses actions; the critic estimates future rewards. Actions doing better than expected become more likely, with an objective that limits excessive changes. Exploration tries other actions, while precise control requires reducing unnecessary randomness.

The first cell sets `num_envs` (parallel experience collection) and `max_iterations` (collection and optimization cycles). The next starts **3–4 min** of computation, including evaluation and export. More environments also increase the data budget per iteration; more iterations do not guarantee a better result.

The curve measures **reward**, not success rate. A separate evaluation measures drawer opening.
""")

## 2. Learn a gesture · 18 min

### 2.1 RL: a privileged expert

**Actual training — follow the notebook output and curve. Isaac Sim may stay still.**

In reinforcement learning (**RL**), a **policy** maps observations to actions. The simulator computes a **reward**: approaching the handle, aligning the gripper, grasping and opening earn reward here; abrupt movements are penalized. The network learns to increase the sum of future rewards.

An **environment** is a simulated copy of the problem; an **episode** is an attempt between resets. The 256 environments collect experience for one shared policy. It receives joint and drawer state and the relative handle position, **not images**.

**PPO** alternates experience collection and weight updates. The actor chooses actions; the critic estimates future rewards. Actions doing better than expected become more likely, with an objective that limits excessive changes. Exploration tries other actions, while precise control requires reducing unnecessary randomness.

The first cell sets `num_envs` (parallel experience collection) and `max_iterations` (collection and optimization cycles). The next starts **3–4 min** of computation, including evaluation and export. More environments also increase the data budget per iteration; more iterations do not guarantee a better result.

The curve measures **reward**, not success rate. A separate evaluation measures drawer opening.

In [ ]:
RL = yaml.safe_load('''
num_envs: 256          # scene copies trained in parallel (64 .. 1024)
max_iterations: 50     # PPO updates (a few seconds per iteration)
''')

In [ ]:
tl("rl_train_start", **RL)
if QUICK: RL.update(num_envs=64, max_iterations=3)
log = run(f"./train.sh rl --num_envs {RL['num_envs']} --max_iterations {RL['max_iterations']}",
          keep=r"Learning iteration", every=10, always=r"evaluating the checkpoints|exported|to make the robot|no model|no checkpoint")
it = [int(m) for m in re.findall(r"Learning iteration (\d+)/", log)]
rw = [float(m) for m in re.findall(r"Mean reward:\s*([-\d.]+)", log)]
if rw:
    plt.plot(it[:len(rw)], rw); plt.xlabel(t("iteration", "iteration")); plt.ylabel(t("mean reward", "mean reward"))
    plt.title(t("your RL training", "your RL training")); plt.show()
tl("rl_train_end", iterations=len(rw))

In [ ]:
T(r"""
### From checkpoint to robot

**First cell: reference — watch the notebook videos.** They come from a longer training run at iterations 0, 50 and 250, not the run you just launched.

**Second cell: live robot — watch Isaac Sim.** It runs **your model** without changing its weights. `checkpoint_source` and `sha256` identify the checkpoint used. The reference model is restored afterwards.
""",
r"""
### From checkpoint to robot

**First cell: reference — watch the notebook videos.** They come from a longer training run at iterations 0, 50 and 250, not the run you just launched.

**Second cell: live robot — watch Isaac Sim.** It runs **your model** without changing its weights. `checkpoint_source` and `sha256` identify the checkpoint used. The reference model is restored afterwards.
""")

### From checkpoint to robot

**First cell: reference — watch the notebook videos.** They come from a longer training run at iterations 0, 50 and 250, not the run you just launched.

**Second cell: live robot — watch Isaac Sim.** It runs **your model** without changing its weights. `checkpoint_source` and `sha256` identify the checkpoint used. The reference model is restored afterwards.

In [ ]:
for e in map(json.loads, open(f"{A}/rl/eval.jsonl")):
    print(t(f"iteration {e['iteration']:4d} | success {e['success_rate']*100:5.1f} % | mean opening {e['mean_max_opening_m']:.2f} m",
            f"iteration {e['iteration']:4d} | success {e['success_rate']*100:5.1f} % | mean opening {e['mean_max_opening_m']:.2f} m"))
for it_ in (0, 50, 250):
    print(t("iteration", "iteration"), it_); display(Video(f"{A}/rl/it_{it_:04d}.mp4", embed=True, width=360))

In [ ]:
mine = sorted(glob.glob(f"{ROOT}/results/train_rl_*/policy/MODEL.json"))
if mine:  # your model: ./train.sh select writes its path + sha256 into team/config.yaml
    run(f"./train.sh select {os.path.relpath(os.path.dirname(os.path.dirname(mine[-1])), ROOT)}")
lab.reset(); r = lab.skill("open_drawer", provider="rl")
ev = r["evidence"]
print("open_drawer(rl) :", r["ok"], "|", r["reason"], t("| model run:", "| model run:"), str(ev.get("checkpoint_sha256", ""))[:12], "-", ev.get("checkpoint_source"))
run("./train.sh select reference open_drawer_rl")   # restore the reference model for subsequent experiments
print(t("→ to make the robot use YOUR RL model later: ./train.sh select results/<run>",
        "→ to make the robot use YOUR RL model later: ./train.sh select results/<run>"))

In [ ]:
T(r"""
`open_drawer` provides three implementations (`provider=` or `team/config.yaml`):

| Provider | Control | Observations |
|---|---|---|
| `rl` | Learned policy | Exact handle position |
| `analytic` | Geometric controller | Exact handle position |
| `visual` | Reference DAgger student | RGB-D and joints |

Image rendering makes `visual` slower.
""",
r"""
`open_drawer` provides three implementations (`provider=` or `team/config.yaml`):

| Provider | Control | Observations |
|---|---|---|
| `rl` | Learned policy | Exact handle position |
| `analytic` | Geometric controller | Exact handle position |
| `visual` | Reference DAgger student | RGB-D and joints |

Image rendering makes `visual` slower.
""")

`open_drawer` provides three implementations (`provider=` or `team/config.yaml`):

| Provider | Control | Observations |
|---|---|---|
| `rl` | Learned policy | Exact handle position |
| `analytic` | Geometric controller | Exact handle position |
| `visual` | Reference DAgger student | RGB-D and joints |

Image rendering makes `visual` slower.

In [ ]:
T(r"""
### 2.2 BC and DAgger: correcting visited states

**Actual training — follow notebook outputs, tables and videos.**

The teacher uses the exact state; the student receives **RGB-D** (colour and depth) **and joints**. **Behaviour cloning (BC)** trains the student to reproduce actions from teacher demonstrations: supervised learning.

When acting, the student may drift into states missing from those demonstrations, especially under perturbations. **DAgger** lets it act, queries teacher corrections in these states, adds them to earlier data, then retrains. Training uses teacher action targets here rather than the RL reward; a DAgger round is not a PPO iteration.

**Your training, about 4 min:** collection → BC → two DAgger rounds → evaluation. The teacher is pretrained, independent of your short RL run.

Two metrics: **success > 20 cm** at least once; **goal ≥ 30 cm** reached. The challenge uses separate criteria.
""",
r"""
### 2.2 BC and DAgger: correcting visited states

**Actual training — follow notebook outputs, tables and videos.**

The teacher uses the exact state; the student receives **RGB-D** (colour and depth) **and joints**. **Behaviour cloning (BC)** trains the student to reproduce actions from teacher demonstrations: supervised learning.

When acting, the student may drift into states missing from those demonstrations, especially under perturbations. **DAgger** lets it act, queries teacher corrections in these states, adds them to earlier data, then retrains. Training uses teacher action targets here rather than the RL reward; a DAgger round is not a PPO iteration.

**Your training, about 4 min:** collection → BC → two DAgger rounds → evaluation. The teacher is pretrained, independent of your short RL run.

Two metrics: **success > 20 cm** at least once; **goal ≥ 30 cm** reached. The challenge uses separate criteria.
""")

### 2.2 BC and DAgger: correcting visited states

**Actual training — follow notebook outputs, tables and videos.**

The teacher uses the exact state; the student receives **RGB-D** (colour and depth) **and joints**. **Behaviour cloning (BC)** trains the student to reproduce actions from teacher demonstrations: supervised learning.

When acting, the student may drift into states missing from those demonstrations, especially under perturbations. **DAgger** lets it act, queries teacher corrections in these states, adds them to earlier data, then retrains. Training uses teacher action targets here rather than the RL reward; a DAgger round is not a PPO iteration.

**Your training, about 4 min:** collection → BC → two DAgger rounds → evaluation. The teacher is pretrained, independent of your short RL run.

Two metrics: **success > 20 cm** at least once; **goal ≥ 30 cm** reached. The challenge uses separate criteria.

In [ ]:
DAGGER = yaml.safe_load('''
num_demonstrations: 32     # teacher demonstrations for BC (multiple of 32)
perturbation:
  slip_prob: 0.7           # probability of handle slip during pulling (0 = never)
  push_rad: 0.35           # arm push during the approach (radians, 0 = none)
dagger_rounds: 2           # 1 or 2 DAgger rounds
''')

In [ ]:
d = DAGGER; pert = d["perturbation"]
if QUICK: d["dagger_rounds"] = 1
stages = ["teacher_eval", "bc", "r1"] + (["r2"] if d["dagger_rounds"] >= 2 else []) + ["report"]
args = (f"--num_envs 32 --bc_batches {max(1, d['num_demonstrations'] // 32)} --round_batches 1 --n_video 4 "
        f"--stages {','.join(stages)} --slip_prob {pert['slip_prob']} --push_rad {pert['push_rad']}"
        + (" --perturb" if pert["slip_prob"] > 0 or pert["push_rad"] > 0 else ""))
tl("dagger_start", **{"demos": d["num_demonstrations"], "rounds": d["dagger_rounds"], **pert})
if QUICK: args += " --bc_steps 300 --round_steps 200 --n_video 1"
log = run(f"./train.sh dagger {args}", keep=r"\] (EVAL|TIMING (train|collect))|\] done", every=1)
runs = sorted(glob.glob(f"{ROOT}/results/train_dagger_*"))
S = json.load(open(f"{runs[-1]}/summary.json")) if runs and os.path.exists(f"{runs[-1]}/summary.json") else None
tl("dagger_end", ok=S is not None)

In [ ]:
def table(S, title):
    print(title)
    print(f"{t('method', 'method'):10s} {t('success > 20 cm', 'success > 20 cm'):>15s} {t('goal ≥ 30 cm', 'goal ≥ 30 cm'):>17s} "
          f"{t('states off the teacher', 'states off the teacher'):>25s} {t('stuck after a slip', 'stuck after a slip'):>24s}")
    for k in ("teacher", "bc", "r1", "r2"):
        if k in S:
            s = S[k]
            print(f"{k:10s} {s.get('success_rate', 0)*100:14.0f} % {s.get('reached_skill_target_0.30m', float('nan'))*100:16.0f} % "
                  f"{s.get('shift_frac_states_off_teacher_support', float('nan'))*100:24.0f} % {s.get('stall_below_0.30m_if_slip', float('nan'))*100:23.0f} %")
if S: table(S, t("YOUR experiment:", "YOUR experiment:"))
table(json.load(open(f"{A}/dagger/summary.json")), t("\nReference (32 demos, perturbations, 128 episodes):", "\nReference (32 demos, perturbations, 128 episodes):"))
vids = sorted(glob.glob(f"{runs[-1]}/videos/*.mp4")) if runs else []
for v in vids[:2]:
    print(os.path.basename(v)); display(Video(v, embed=True, width=560))
print(t("\"success\" = drawer opened past 20 cm at least once; \"goal\" = the 30 cm opening aimed at by the skill.",
        "\"success\" = drawer opened past 20 cm at least once; \"goal\" = the 30 cm opening aimed at by the skill."))
print(t("teacher / BC / DAgger r1 / r2 on the same episodes (reference). The % shown is the success > 20 cm;",
        "teacher / BC / DAgger r1 / r2 on the same episodes (reference). The % shown is the success > 20 cm;"))
print(t("drawer colour: green ≥ 30 cm, orange 20-30 cm, red < 20 cm.", "drawer colour: green ≥ 30 cm, orange 20-30 cm, red < 20 cm."))
display(Video(f"{A}/dagger/compare_teacher_bc_r1_r2.mp4", embed=True, width=640))

In [ ]:
T(r"""
### Test your student

**Live robot — watch Isaac Sim, then the notebook images.**

Compare BC, r1 and r2: opening, out-of-distribution states, stalls after slipping. Reference results and your run are shown separately.

The cell loads **your student**, runs it, then restores the reference. The skill response identifies the execution; the image lets you inspect the outcome.
""",
r"""
### Test your student

**Live robot — watch Isaac Sim, then the notebook images.**

Compare BC, r1 and r2: opening, out-of-distribution states, stalls after slipping. Reference results and your run are shown separately.

The cell loads **your student**, runs it, then restores the reference. The skill response identifies the execution; the image lets you inspect the outcome.
""")

### Test your student

**Live robot — watch Isaac Sim, then the notebook images.**

Compare BC, r1 and r2: opening, out-of-distribution states, stalls after slipping. Reference results and your run are shown separately.

The cell loads **your student**, runs it, then restores the reference. The skill response identifies the execution; the image lets you inspect the outcome.

In [ ]:
mine = sorted(glob.glob(f"{ROOT}/results/train_dagger_*/policy/MODEL.json"))
if mine:
    run(f"./train.sh select {os.path.relpath(os.path.dirname(os.path.dirname(mine[-1])), ROOT)}")
lab.reset(); r = lab.skill("open_drawer_student")
ev = r["evidence"]
print(r["reason"], t("| model run:", "| model run:"), str(ev.get("checkpoint_sha256", ""))[:12], "-", ev.get("checkpoint_source"))
lab.show(lab.observe(), t("is the drawer open? your call", "is the drawer open? your call"))
run("./train.sh select reference open_drawer_student")   # restore the reference student for subsequent experiments
print(t("→ to make the robot (and your Brain) use YOUR student: ./train.sh select results/<run>",
        "→ to make the robot (and your Brain) use YOUR student: ./train.sh select results/<run>"))

In [ ]:
T(r"""
## 3. RAM insertion · 7 min

**Reference, then live robot in the notebook: the RAM scene has no dedicated Isaac window.**

The stick starts in the gripper; the case changes pose between episodes. Insertion clearance is below a millimetre.

**Reference:** two BC → DAgger chains, evaluated on 64 fixed held-out episodes, with 95% Wilson intervals. Equal-data controls and the shipped student are evaluated separately over 192 episodes. No retraining in this block.
""",
r"""
## 3. RAM insertion · 7 min

**Reference, then live robot in the notebook: the RAM scene has no dedicated Isaac window.**

The stick starts in the gripper; the case changes pose between episodes. Insertion clearance is below a millimetre.

**Reference:** two BC → DAgger chains, evaluated on 64 fixed held-out episodes, with 95% Wilson intervals. Equal-data controls and the shipped student are evaluated separately over 192 episodes. No retraining in this block.
""")

## 3. RAM insertion · 7 min

**Reference, then live robot in the notebook: the RAM scene has no dedicated Isaac window.**

The stick starts in the gripper; the case changes pose between episodes. Insertion clearance is below a millimetre.

**Reference:** two BC → DAgger chains, evaluated on 64 fixed held-out episodes, with 95% Wilson intervals. Equal-data controls and the shipped student are evaluated separately over 192 episodes. No retraining in this block.

In [ ]:
R = json.load(open(f"{A}/ram/chain_results.json"))
def row(r):
    return f"{r['success']:3d}/{r['n']} = {r['rate_pct']:5.1f} %  [{r['ci95_pct'][0]:.0f}-{r['ci95_pct'][1]:.0f}]"
print(t("stick seated, 64 fixed evaluation episodes (95 % confidence interval):",
        "stick seated, 64 fixed evaluation episodes (95 % confidence interval):"))
print(f"{t('stage', 'stage'):6s} {t('data', 'data'):>8s}   {t('chain 1 (teacher v1)', 'chain 1 (teacher v1)'):32s} {t('chain 2 (teacher v2)', 'chain 2 (teacher v2)'):32s}")
c1 = {r["stage"]: r for r in R["chain1"]["rounds"]}
for r in R["chain2"]["rounds"]:
    print(f"{r['stage']:6s} {r['samples']:8d}   {row(c1[r['stage']]) if r['stage'] in c1 else '—':32s} {row(r)}")
m = R["chain2"]["masked_images"]
print(t("masked (black) images:", "masked (black) images:"), f"BC {row(m['bc'])} · r2 {row(m['r2'])}")
k = R["controls"]
print(t("\nsame data volume (192 episodes):", "\nsame data volume (192 episodes):"), f"DAgger r4 {row(k['same_volume']['dagger_r4'])} · "
      + t("BC on 320 demos", "BC on 320 demos"), row(k['same_volume']['bc_320_demos']))
print(t("same data as r2: fresh student retrained on the whole aggregate", "same data as r2: fresh student retrained on the whole aggregate"),
      row(k['retrain_from_scratch']['fresh_on_aggregate_r2']), t("· fine-tuned r2", "· fine-tuned r2"), row(k['retrain_from_scratch']['fine_tuned_r2']))
s = R["shipped"]
print(t(f"\nshipped student ({s['stage']}, sha256 {s['sha256'][:12]}…):", f"\nshipped student ({s['stage']}, sha256 {s['sha256'][:12]}…):"),
      f"{row(s['eval_192'])} ;", t("through the scene server", "through the scene server"), row(s['through_the_skill_server']))

In [ ]:
T(r"""
### What the measurements show

- Teacher v1: **60/60** alone, but discontinuous sub-millimetre corrections. Its student goes from **48 to 9 to 17%** (BC, r1, r2).
- Teacher v2, gradual corrections: **64 → 14 → 88%** (BC, r1, r5). DAgger does not improve every round.
- Equal data volume: **83.9% DAgger versus 78.6% BC**, over 192 episodes. Overlapping intervals alone do not establish statistical significance or its absence.
- Same r2 data: **73%** retraining from scratch, **44%** with the tested fine-tuning. Black images: **0/64** for both BC and r2.

**Execution, about 15 s:** human HD view on the left; 128 px cameras on the right. The Brain receives these images; the policy downsamples its RGB-D inputs to 64 px. Neither receives the HD view.
""",
r"""
### What the measurements show

- Teacher v1: **60/60** alone, but discontinuous sub-millimetre corrections. Its student goes from **48 to 9 to 17%** (BC, r1, r2).
- Teacher v2, gradual corrections: **64 → 14 → 88%** (BC, r1, r5). DAgger does not improve every round.
- Equal data volume: **83.9% DAgger versus 78.6% BC**, over 192 episodes. Overlapping intervals alone do not establish statistical significance or its absence.
- Same r2 data: **73%** retraining from scratch, **44%** with the tested fine-tuning. Black images: **0/64** for both BC and r2.

**Execution, about 15 s:** human HD view on the left; 128 px cameras on the right. The Brain receives these images; the policy downsamples its RGB-D inputs to 64 px. Neither receives the HD view.
""")

### What the measurements show

- Teacher v1: **60/60** alone, but discontinuous sub-millimetre corrections. Its student goes from **48 to 9 to 17%** (BC, r1, r2).
- Teacher v2, gradual corrections: **64 → 14 → 88%** (BC, r1, r5). DAgger does not improve every round.
- Equal data volume: **83.9% DAgger versus 78.6% BC**, over 192 episodes. Overlapping intervals alone do not establish statistical significance or its absence.
- Same r2 data: **73%** retraining from scratch, **44%** with the tested fine-tuning. Black images: **0/64** for both BC and r2.

**Execution, about 15 s:** human HD view on the left; 128 px cameras on the right. The Brain receives these images; the policy downsamples its RGB-D inputs to 64 px. Neither receives the HD view.

In [ ]:
tl("ram_start")
pc = Lab("http://127.0.0.1:8766")                   # PC scene (without a dedicated window)
pc.start_run(f"notebook_ram_{int(time.time())}", "insert the RAM stick", reset=True)   # new episode, different case position
seen = t("what the student sees", "what the student sees")
pc.show_scene(t("before: the stick is in the gripper", "before: the stick is in the gripper"), seen)
r = pc.skill("insert_ram_student")
ev = r["evidence"]
print(r["reason"], t("| model run:", "| model run:"), str(ev.get("checkpoint_sha256", ""))[:12], "-", ev.get("checkpoint_source"))
pc.show_scene(t("after: is the stick seated? your call", "after: is the stick seated? your call"), seen)
v = pc.end_run().get("video")
if v: display(Video(v, embed=True, width=900))

In [ ]:
T(r"""
### Visual verdict

Seated, not seated, or inconclusive? Compare the HD view with the 128 px images alone **before running the next cell**.

It reveals the evaluator's ground truth, available here for analysis, **not for the challenge Brain**.
""",
r"""
### Visual verdict

Seated, not seated, or inconclusive? Compare the HD view with the 128 px images alone **before running the next cell**.

It reveals the evaluator's ground truth, available here for analysis, **not for the challenge Brain**.
""")

### Visual verdict

Seated, not seated, or inconclusive? Compare the HD view with the 128 px images alone **before running the next cell**.

It reveals the evaluator's ground truth, available here for analysis, **not for the challenge Brain**.

In [ ]:
import urllib.request
g = json.load(urllib.request.urlopen("http://127.0.0.1:8766/eval/grade"))   # evaluator ground truth for learning here; not an input to your Brain
print(t("evaluator's verdict:", "evaluator's verdict:"),
      t("YES, stick seated", "YES, stick seated") if g["ram_seated"] else t(f"NO, stick not seated ({g['where']})", f"NO, stick not seated ({g['where']})"))
tl("ram_end", seated=bool(g["ram_seated"]))

In [ ]:
T(r"""
## 4. Choose, predict and decide · 9 min

### 4.1 VLA: SmolVLA on cube grasping

**Reference — notebook plots and videos; no VLA training is launched here.** A Vision–Language–Action model uses vision and an instruction to produce robot commands.

Fine-tuning with 10, 40 or 100 demonstrations; variants in training duration and relative actions (displacements rather than absolute positions). Each configuration is evaluated over **20 episodes with the same seeds**. Compare the scores and videos.
""",
r"""
## 4. Choose, predict and decide · 9 min

### 4.1 VLA: SmolVLA on cube grasping

**Reference — notebook plots and videos; no VLA training is launched here.** A Vision–Language–Action model uses vision and an instruction to produce robot commands.

Fine-tuning with 10, 40 or 100 demonstrations; variants in training duration and relative actions (displacements rather than absolute positions). Each configuration is evaluated over **20 episodes with the same seeds**. Compare the scores and videos.
""")

## 4. Choose, predict and decide · 9 min

### 4.1 VLA: SmolVLA on cube grasping

**Reference — notebook plots and videos; no VLA training is launched here.** A Vision–Language–Action model uses vision and an instruction to produce robot commands.

Fine-tuning with 10, 40 or 100 demonstrations; variants in training duration and relative actions (displacements rather than absolute positions). Each configuration is evaluated over **20 episodes with the same seeds**. Compare the scores and videos.

In [ ]:
rows = [(t("10 demos", "10 demos"), "p2_d10"), (t("40 demos", "40 demos"), "p2_d40"), (t("100 demos", "100 demos"), "p2_d100"),
        (t("40 demos, 10 epochs", "40 demos, 10 epochs"), "p2_smolvla_isaac_d40e"),
        (t("40 demos, relative actions", "40 demos, relative actions"), "p2_d40rel_003000")]
vals = [(lbl, 100 * s["successes"] / s["episodes"]) for lbl, f in rows if os.path.exists(f"{A}/vla/{f}.json")
        for s in [json.load(open(f"{A}/vla/{f}.json"))]]
plt.barh([v[0] for v in vals], [v[1] for v in vals]); plt.axvline(50, ls="--", c="k"); plt.xlabel(t("grasp success (%)", "grasp success (%)")); plt.show()
print(t("a success:", "a success:")); display(Video(f"{A}/vla/success.mp4", embed=True, width=320))
print(t("a failure:", "a failure:"));     display(Video(f"{A}/vla/failure.mp4", embed=True, width=320))
vr = json.load(open(f"{A}/vla/vla_retry.json"))
print(t(f"VLA + \"check and retry\" (3 attempts): {vr['with_retry_cube_placed']}/{vr['episodes']} cubes put away",
        f"VLA + \"check and retry\" (3 attempts): {vr['with_retry_cube_placed']}/{vr['episodes']} cubes put away"))

In [ ]:
T(r"""
In these trials, gripper closure error is **1–3 cm for the VLA**, versus **under 3 mm for the teacher**. The tested configurations do not make grasping reliable.

### 4.2 World model: prediction and plan selection

**Reference — read the notebook results.** A policy chooses an action; a dynamics model predicts a future state.

Three candidate plans, their predicted and measured openings; then a drawer dynamics model compared with the prediction “it does not move”.
""",
r"""
In these trials, gripper closure error is **1–3 cm for the VLA**, versus **under 3 mm for the teacher**. The tested configurations do not make grasping reliable.

### 4.2 World model: prediction and plan selection

**Reference — read the notebook results.** A policy chooses an action; a dynamics model predicts a future state.

Three candidate plans, their predicted and measured openings; then a drawer dynamics model compared with the prediction “it does not move”.
""")

In these trials, gripper closure error is **1–3 cm for the VLA**, versus **under 3 mm for the teacher**. The tested configurations do not make grasping reliable.

### 4.2 World model: prediction and plan selection

**Reference — read the notebook results.** A policy chooses an action; a dynamics model predicts a future state.

Three candidate plans, their predicted and measured openings; then a drawer dynamics model compared with the prediction “it does not move”.

In [ ]:
wm = json.load(open(f"{A}/world_model/plan_choice.json"))
print(t("1) choosing between 3 plans by \"imagining\" them:", "1) choosing between 3 plans by \"imagining\" them:"))
for c_, p_, r_ in zip(wm["candidates"], wm["predicted_final_opening_m"], wm["real_final_opening_m"]):
    print(f"   {c_:22s}", t(f"predicted {p_:.2f} m | real {r_:.2f} m", f"predicted {p_:.2f} m | real {r_:.2f} m"))
print(t("   chosen by the model:", "   chosen by the model:"), wm["chosen_by_model"], t("| best in reality:", "| best in reality:"), wm["best_in_reality"])
dm = json.load(open(f"{A}/world_model/drawer_model.json"))
print(t(f"2) drawer model: error {dm['val_mae_m']['+2s']*100:.1f} cm at 2 s (against {dm['naive_mae_m']['+2s']*100:.1f} cm for \"it does not move\")",
        f"2) drawer model: error {dm['val_mae_m']['+2s']*100:.1f} cm at 2 s (against {dm['naive_mae_m']['+2s']*100:.1f} cm for \"it does not move\")"))

In [ ]:
T(r"""
The model predicts better than the baseline, but changes **0 of 159 decisions over 50 episodes** in the tested loop. Waiting 2 s gave the same decision; this model was not retained among the tools.

### 4.3 Brain: orchestration and verification

In the challenge, the Brain is your decision logic in **`team/brain.py`**; an LLM is optional.

**Reference:** a coding agent calls `observe`, `open_drawer` and `close_drawer` through MCP `vinci-robot`. It observes images and robot state, without simulator ground truth on the task. Some skills nevertheless use privileged control.

The table separates task success from the accuracy of the agent's claim.
""",
r"""
The model predicts better than the baseline, but changes **0 of 159 decisions over 50 episodes** in the tested loop. Waiting 2 s gave the same decision; this model was not retained among the tools.

### 4.3 Brain: orchestration and verification

In the challenge, the Brain is your decision logic in **`team/brain.py`**; an LLM is optional.

**Reference:** a coding agent calls `observe`, `open_drawer` and `close_drawer` through MCP `vinci-robot`. It observes images and robot state, without simulator ground truth on the task. Some skills nevertheless use privileged control.

The table separates task success from the accuracy of the agent's claim.
""")

The model predicts better than the baseline, but changes **0 of 159 decisions over 50 episodes** in the tested loop. Waiting 2 s gave the same decision; this model was not retained among the tools.

### 4.3 Brain: orchestration and verification

In the challenge, the Brain is your decision logic in **`team/brain.py`**; an LLM is optional.

**Reference:** a coding agent calls `observe`, `open_drawer` and `close_drawer` through MCP `vinci-robot`. It observes images and robot state, without simulator ground truth on the task. Some skills nevertheless use privileged control.

The table separates task success from the accuracy of the agent's claim.

In [ ]:
ref = json.load(open(f"{A}/brain/reference_runs.json"))
print(t("model:", "model:"), ", ".join(ref['summary']['models']), "| runs :", ref['summary']['runs'], t("| task succeeded:", "| task succeeded:"),
      ref['summary']['task_success'], t("| agent's claim = truth:", "| agent's claim = truth:"), ref['summary']['claims_match_truth'])
for r_ in ref["runs"][:3]:
    print(f"  run {r_['run']}: {' → '.join(r_['tools_called'])} |",
          t(f"opened {r_['evaluator']['max_opening_m']} m, final {r_['evaluator']['final_opening_m']} m | the agent says {r_['brain_claim']}",
            f"opened {r_['evaluator']['max_opening_m']} m, final {r_['evaluator']['final_opening_m']} m | the agent says {r_['brain_claim']}"))
for k_, t_ in ref.get("verification_tests", {}).items():
    ok_ = sum(bool(x["claim_matches_truth"]) for x in t_["runs"])
    print(t(f"test \"{k_}\": claim = truth in {ok_}/{len(t_['runs'])} runs.", f"test \"{k_}\": claim = truth in {ok_}/{len(t_['runs'])} runs."), t_["note"])

In [ ]:
T(r"""
### Mission: execute, verify, recover

**Live robot — watch Isaac Sim, then read the notebook verdict. No retraining here.**

The reference agent struggles to distinguish a 27 cm opening from 30 cm in the image: “cannot confirm” remains a useful answer.

The visual student attempts to open the drawer. Its `ok` response confirms execution, not opening. We therefore check the opening before placing the cube. If it is insufficient, the reference RL policy takes over once.

**This recovery is an explicit Python rule, not an LLM decision.** The check reads the exact simulator state; the RL fallback, `pick_and_place` and `close_drawer` also use privileged state. This is not an images-only mission. Successful recovery does not count as a visual-student success.

Before picking, the arm moves to a clear waypoint (`go_pick_start`) to avoid crossing the open drawer. The mission stops if a prerequisite remains unmet. The final verdict checks that the cube is put away and the drawer closed.

`TASK FAILED` is a trial outcome; a Python exception indicates an execution problem.
""",
r"""
### Mission: execute, verify, recover

**Live robot — watch Isaac Sim, then read the notebook verdict. No retraining here.**

The reference agent struggles to distinguish a 27 cm opening from 30 cm in the image: “cannot confirm” remains a useful answer.

The visual student attempts to open the drawer. Its `ok` response confirms execution, not opening. We therefore check the opening before placing the cube. If it is insufficient, the reference RL policy takes over once.

**This recovery is an explicit Python rule, not an LLM decision.** The check reads the exact simulator state; the RL fallback, `pick_and_place` and `close_drawer` also use privileged state. This is not an images-only mission. Successful recovery does not count as a visual-student success.

Before picking, the arm moves to a clear waypoint (`go_pick_start`) to avoid crossing the open drawer. The mission stops if a prerequisite remains unmet. The final verdict checks that the cube is put away and the drawer closed.

`TASK FAILED` is a trial outcome; a Python exception indicates an execution problem.
""")

### Mission: execute, verify, recover

**Live robot — watch Isaac Sim, then read the notebook verdict. No retraining here.**

The reference agent struggles to distinguish a 27 cm opening from 30 cm in the image: “cannot confirm” remains a useful answer.

The visual student attempts to open the drawer. Its `ok` response confirms execution, not opening. We therefore check the opening before placing the cube. If it is insufficient, the reference RL policy takes over once.

**This recovery is an explicit Python rule, not an LLM decision.** The check reads the exact simulator state; the RL fallback, `pick_and_place` and `close_drawer` also use privileged state. This is not an images-only mission. Successful recovery does not count as a visual-student success.

Before picking, the arm moves to a clear waypoint (`go_pick_start`) to avoid crossing the open drawer. The mission stops if a prerequisite remains unmet. The final verdict checks that the cube is put away and the drawer closed.

`TASK FAILED` is a trial outcome; a Python exception indicates an execution problem.

In [ ]:
tl("first_agent")
lab.reset()
def call_skill(name, **kw):
    result = lab.skill(name, **kw)
    print(f"{'OK   ' if result['ok'] else t('FAIL ', 'FAIL ')} {name}({kw}) : {result['reason']}")
    return result

# Reference for reproducible recovery, even after your training experiments.
run("./train.sh select reference open_drawer_rl")
call_skill("open_drawer_student")
opening = lab.observe()["drawer_opening_m"]
print(t("Opening after the student:", "Opening after the student:"), opening, "m")
if opening < 0.18:
    print(t("RECOVERY: insufficient opening, calling the reference RL policy.",
            "RECOVERY: insufficient opening, calling the reference RL policy."))
    call_skill("open_drawer", provider="rl")
    opening = lab.observe()["drawer_opening_m"]

if opening >= 0.18:
    ready = call_skill("go_pick_start")
else:
    ready = {"ok": False}

if opening >= 0.18 and ready["ok"]:
    placed = call_skill("pick_and_place", obj="cube", target="top_drawer")
    if placed["ok"] and lab.success().get("cube_in_drawer"):
        call_skill("close_drawer")
    else:
        print(t("STOP: placement not confirmed; leaving the drawer open.",
                "STOP: placement not confirmed; leaving the drawer open."))
else:
    print(t("STOP: opening or starting pose insufficient; placement skipped.",
            "STOP: opening or starting pose insufficient; placement skipped."))

w = lab.success()
print(t("the world's verdict:", "the world's verdict:"), t("TASK SUCCEEDED", "TASK SUCCEEDED") if w.get("success") else t("TASK FAILED", "TASK FAILED"),
      "|", {k: v for k, v in w.items() if k != "success"})

In [ ]:
T(r"""
# Guided lab complete. The team challenge starts now.

Open **Challenge sheet**, **VS Code** and **Terminal**. The challenge runs in `team/`. The challenge sheet is in French.

**Evaluation: choose just one option in the terminal.**
- **Without video (recommended)**: `./evaluate.sh`, about **1–2 min** for 8 episodes.
- **With two videos (optional)**: `./evaluate.sh --videos 2`, about **9–10 min in our test**. The desktop may become less responsive while recording; let the command finish.

Both options compute the score. You can skip videos and submit without recording any. Times depend on your settings and selected provider.

1. Run your chosen option; record score, success and simulated time.
2. Enable `verification.enabled` in `team/config.yaml`, then evaluate again. The additional points appear in `identified`; `final_state_verified` is a next improvement to explore.
3. Choose an improvement from the sheet, test it, then `./submit.sh`.

Run the last cell: it displays `brain.py` and marks the start of the challenge.
""",
r"""
# Guided lab complete. The team challenge starts now.

Open **Challenge sheet**, **VS Code** and **Terminal**. The challenge runs in `team/`. The challenge sheet is in French.

**Evaluation: choose just one option in the terminal.**
- **Without video (recommended)**: `./evaluate.sh`, about **1–2 min** for 8 episodes.
- **With two videos (optional)**: `./evaluate.sh --videos 2`, about **9–10 min in our test**. The desktop may become less responsive while recording; let the command finish.

Both options compute the score. You can skip videos and submit without recording any. Times depend on your settings and selected provider.

1. Run your chosen option; record score, success and simulated time.
2. Enable `verification.enabled` in `team/config.yaml`, then evaluate again. The additional points appear in `identified`; `final_state_verified` is a next improvement to explore.
3. Choose an improvement from the sheet, test it, then `./submit.sh`.

Run the last cell: it displays `brain.py` and marks the start of the challenge.
""")

# Guided lab complete. The team challenge starts now.

Open **Challenge sheet**, **VS Code** and **Terminal**. The challenge runs in `team/`. The challenge sheet is in French.

**Evaluation: choose just one option in the terminal.**
- **Without video (recommended)**: `./evaluate.sh`, about **1–2 min** for 8 episodes.
- **With two videos (optional)**: `./evaluate.sh --videos 2`, about **9–10 min in our test**. The desktop may become less responsive while recording; let the command finish.

Both options compute the score. You can skip videos and submit without recording any. Times depend on your settings and selected provider.

1. Run your chosen option; record score, success and simulated time.
2. Enable `verification.enabled` in `team/config.yaml`, then evaluate again. The additional points appear in `identified`; `final_state_verified` is a next improvement to explore.
3. Choose an improvement from the sheet, test it, then `./submit.sh`.

Run the last cell: it displays `brain.py` and marks the start of the challenge.

In [ ]:
tl("challenge_start")
print(open(os.path.join(ROOT, "team", "brain.py")).read())